[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day23_neural_networks/day23_notebook.ipynb)

# Day 23 / 42: Neural Networks
### #42DaysOfML Challenge

---

## What You'll Learn
- What a neural network is and how it's structured (layers, neurons, weights, biases)
- How activation functions work and why they matter
- How a forward pass computes a prediction from scratch
- How backpropagation updates weights using gradients
- How to train a neural network from scratch in pure NumPy (XOR problem)
- How to build and train one using Keras for MNIST digit classification
- A real production problem and how engineers handle it

---

**Prerequisites:** NumPy, basic Python. No deep learning experience needed.

**Dataset used:** XOR (hand-crafted), MNIST (Keras built-in)


## Part 1: The Concept

A neural network is a chain of mathematical operations applied to your input data, layer by layer, until you get a prediction.

Each layer consists of:
- **Neurons** (nodes): one per output value of that layer
- **Weights (W)**: how much each input contributes to each neuron
- **Biases (b)**: a constant offset added before the activation
- **Activation function**: a non-linear function applied to the weighted sum

The computation at each layer:  
`Z = X @ W + b`  
`A = activation(Z)`

Without activation functions, stacking layers does nothing (multiple linear operations collapse into one linear operation). Activations introduce the non-linearity that lets networks learn complex patterns.

---

### The Analogy

Think of a neural network as a series of filters at an airport security check. Each filter (layer) looks at the input in a different way, passes along what matters, and discards noise. The final filter gives you a decision: let through or stop.

---

### Network Architecture (3-Layer Example)

```
Input Layer    Hidden Layer 1    Hidden Layer 2    Output Layer
  [X1]  ─────────►  [N1]  ────────► [N3]  ──────►  [Output]
  [X2]  ─────────►  [N2]  ────────► [N4]  ──────►
  [X3]  ─────────►        ────────►        ──────►
```

Every arrow carries a weight. The network learns by adjusting those weights.

In [ ]:
# Install dependencies (only needed if running locally without them)
# !pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('NumPy version:', np.__version__)
print('Setup complete.')

## Part 2: Activation Functions

Activation functions decide how much a neuron "fires". Three you'll see constantly:

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| Sigmoid | 1/(1+e^-x) | (0, 1) | Binary classification output |
| ReLU | max(0, x) | [0, ∞) | Hidden layers (most common) |
| Tanh | (e^x - e^-x)/(e^x + e^-x) | (-1, 1) | Hidden layers, RNNs |

**Why not sigmoid everywhere?** For deep networks, sigmoid causes the *vanishing gradient problem* — gradients become so small during backpropagation that weights in early layers barely update. ReLU avoids this.

In [ ]:
# Define activation functions from scratch
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

# Derivatives (needed for backpropagation)
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu_deriv(x):
    return (x > 0).astype(float)

# Visualise all three
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Activation Functions', fontsize=14, fontweight='bold')

# Sigmoid
axes[0].plot(x, sigmoid(x), color='#0066CC', linewidth=2, label='sigmoid(x)')
axes[0].plot(x, sigmoid_deriv(x), color='#FF6600', linewidth=2, linestyle='--', label="sigmoid'(x)")
axes[0].set_title('Sigmoid', fontweight='bold')
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)
axes[0].legend()
axes[0].set_ylim(-0.2, 1.2)
axes[0].grid(True, alpha=0.3)

# ReLU
axes[1].plot(x, relu(x), color='#00AA44', linewidth=2, label='relu(x)')
axes[1].plot(x, relu_deriv(x), color='#FF6600', linewidth=2, linestyle='--', label="relu'(x)")
axes[1].set_title('ReLU', fontweight='bold')
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Tanh
axes[2].plot(x, tanh(x), color='#9900CC', linewidth=2, label='tanh(x)')
axes[2].set_title('Tanh', fontweight='bold')
axes[2].axhline(y=0, color='k', linewidth=0.5)
axes[2].axvline(x=0, color='k', linewidth=0.5)
axes[2].legend()
axes[2].set_ylim(-1.2, 1.2)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activation_functions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Sigmoid max gradient:', round(sigmoid_deriv(np.array([0.0]))[0], 4), '(at x=0)')
print('ReLU gradient:', 'exactly 1 for x>0, 0 for x<0')
print('Notice: sigmoid gradient is always < 0.25 — this kills gradients in deep nets')

## Part 3: Forward Pass — How a Prediction is Computed

The forward pass takes input data X and pushes it through every layer to produce a prediction.

For each layer:  
1. `Z = X @ W + b` (linear combination)
2. `A = activation(Z)` (non-linearity)

The output of one layer becomes the input to the next.

In [ ]:
# Forward pass — manual implementation
# Problem: XOR gate (can't be solved with a single linear model)
# [0,0] -> 0, [0,1] -> 1, [1,0] -> 1, [1,1] -> 0

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([[0], [1], [1], [0]])

# Architecture: 2 inputs -> 4 hidden neurons -> 1 output
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.1   # shape (input_size, hidden_size)
b1 = np.zeros((1, 4))               # shape (1, hidden_size)
W2 = np.random.randn(4, 1) * 0.1   # shape (hidden_size, output_size)
b2 = np.zeros((1, 1))               # shape (1, output_size)

print('=== FORWARD PASS ===')
print(f'Input X shape: {X.shape}')
print(f'W1 shape: {W1.shape} | b1 shape: {b1.shape}')
print(f'W2 shape: {W2.shape} | b2 shape: {b2.shape}')
print()

# Layer 1: hidden
Z1 = X @ W1 + b1        # linear step
A1 = sigmoid(Z1)         # activation
print(f'Z1 shape: {Z1.shape} (4 samples x 4 hidden neurons)')
print(f'A1 shape: {A1.shape}')

# Layer 2: output
Z2 = A1 @ W2 + b2
A2 = sigmoid(Z2)         # sigmoid for binary output
print(f'Z2 shape: {Z2.shape}')
print(f'A2 (predictions) shape: {A2.shape}')
print()
print('Initial predictions (untrained network):')
for i, (xi, yi, pred) in enumerate(zip(X, y.flatten(), A2.flatten())):
    print(f'  Input {xi} | Expected {yi} | Got {pred:.4f}')

## Part 4: Loss Function

The loss tells you how wrong the network is. For binary classification, we use **Binary Cross-Entropy**:

`L = -mean(y * log(ŷ) + (1-y) * log(1-ŷ))`

When prediction = label, loss is close to 0. When far off, loss is high. Training minimises this loss.

In [ ]:
def binary_crossentropy(y_true, y_pred):
    epsilon = 1e-8  # prevents log(0)
    return -np.mean(
        y_true * np.log(y_pred + epsilon) + 
        (1 - y_true) * np.log(1 - y_pred + epsilon)
    )

initial_loss = binary_crossentropy(y, A2)
print(f'Initial loss (random weights): {initial_loss:.4f}')
print(f'A perfect model would have loss close to 0')
print(f'Random chance on balanced binary = ~0.69 (log 2)')
print()

# Show what the loss landscape looks like for a single prediction
pred_range = np.linspace(0.001, 0.999, 200)

loss_when_y1 = -np.log(pred_range)           # y=1, we want pred close to 1
loss_when_y0 = -np.log(1 - pred_range)       # y=0, we want pred close to 0

plt.figure(figsize=(8, 4))
plt.plot(pred_range, loss_when_y1, label='Loss when y=1', color='#0066CC', linewidth=2)
plt.plot(pred_range, loss_when_y0, label='Loss when y=0', color='#CC0000', linewidth=2)
plt.xlabel('Predicted probability')
plt.ylabel('Loss')
plt.title('Binary Cross-Entropy Loss', fontweight='bold')
plt.legend()
plt.ylim(0, 5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('The network gets penalised heavily for being confidently wrong')

## Part 5: Backpropagation — How the Network Learns

Backpropagation computes how much each weight contributed to the loss, then adjusts weights in the direction that reduces loss (gradient descent).

Three steps every training iteration:
1. **Forward pass**: compute predictions
2. **Compute loss**: measure how wrong the predictions are
3. **Backward pass**: compute gradients using the chain rule, update weights

Weight update rule: `W = W - learning_rate * dL/dW`

In [ ]:
# Full training loop — XOR problem from scratch
np.random.seed(42)

# Reinitialise weights
W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.1
b2 = np.zeros((1, 1))

learning_rate = 0.5
epochs = 10000
loss_history = []

for epoch in range(epochs):
    
    # === FORWARD PASS ===
    Z1 = X @ W1 + b1
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    
    # === COMPUTE LOSS ===
    loss = binary_crossentropy(y, A2)
    loss_history.append(loss)
    
    # === BACKWARD PASS (chain rule) ===
    # Output layer gradients
    dA2 = -(y / (A2 + 1e-8) - (1 - y) / (1 - A2 + 1e-8))
    dZ2 = dA2 * sigmoid_deriv(Z2)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    
    # Hidden layer gradients
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * sigmoid_deriv(Z1)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    # === UPDATE WEIGHTS ===
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

# Final predictions
print('=== TRAINING COMPLETE ===')
print(f'Final loss: {loss_history[-1]:.6f}')
print(f'Initial loss: {loss_history[0]:.4f}')
print()
print('Predictions after training:')
for i, (xi, yi, pred) in enumerate(zip(X, y.flatten(), A2.flatten())):
    status = 'CORRECT' if round(pred) == yi else 'WRONG'
    print(f'  Input {xi} | Expected {yi} | Got {pred:.4f} | {status}')

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='#0066CC', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Time (XOR Problem)', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print('A straight line (logistic regression) cannot solve XOR.')
print('The hidden layer learned a non-linear decision boundary that can.')

## Part 6: Visualising What the Hidden Layer Learned

In [ ]:
# Show the decision boundary the network learned
# Create a fine mesh of input points
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

# Forward pass on grid
Z1_grid = grid @ W1 + b1
A1_grid = sigmoid(Z1_grid)
Z2_grid = A1_grid @ W2 + b2
A2_grid = sigmoid(Z2_grid).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, A2_grid, levels=50, cmap='RdBu', alpha=0.8)
plt.colorbar(label='Output probability')

# Plot the 4 XOR points
colors = ['red' if label == 0 else 'blue' for label in y.flatten()]
labels_text = ['0 (XOR=0)', '1 (XOR=1)']
for i, (xi, yi_val, col) in enumerate(zip(X, y.flatten(), colors)):
    plt.scatter(xi[0], xi[1], c=col, s=200, zorder=5, edgecolors='white', linewidths=2)
    plt.annotate(f'[{xi[0]},{xi[1]}]→{yi_val}', 
                 (xi[0]+0.05, xi[1]+0.05), fontsize=12, fontweight='bold')

plt.title('Decision Boundary Learned by Neural Network (XOR)', fontweight='bold')
plt.xlabel('Input X1')
plt.ylabel('Input X2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('decision_boundary.png', dpi=100, bbox_inches='tight')
plt.show()
print('Blue region: network predicts class 1')
print('Red region:  network predicts class 0')
print('The non-linear boundary correctly separates the XOR classes')

## Part 7: Build the Same Network in Keras (MNIST)

Now that you understand every operation happening inside, we'll use Keras to build a network on a real dataset: MNIST handwritten digits.

The architecture:
- Input: 784 pixels (28x28 flattened)
- Hidden 1: 128 neurons, ReLU
- Hidden 2: 64 neurons, ReLU
- Output: 10 neurons, Softmax (one per digit 0-9)

In [ ]:
# Keras MNIST neural network
try:
    import tensorflow as tf
    from tensorflow import keras
    print(f'TensorFlow version: {tf.__version__}')
    TF_AVAILABLE = True
except ImportError:
    print('TensorFlow not installed.')
    print('Run: pip install tensorflow')
    print('Showing code structure only — run in Colab for full execution')
    TF_AVAILABLE = False

In [ ]:
if TF_AVAILABLE:
    # Load and preprocess MNIST
    (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
    
    print(f'Train shape: {X_train.shape}')
    print(f'Test shape:  {X_test.shape}')
    print(f'Pixel value range before normalisation: {X_train.min()} to {X_train.max()}')
    
    # Normalise pixel values from [0,255] to [0,1]
    X_train = X_train / 255.0
    X_test  = X_test  / 255.0
    
    # Flatten 28x28 images into 784-length vectors
    X_train_flat = X_train.reshape(-1, 784)
    X_test_flat  = X_test.reshape(-1, 784)
    
    print(f'Pixel value range after normalisation: {X_train_flat.min()} to {X_train_flat.max()}')
    print(f'Flattened train shape: {X_train_flat.shape}')
    
    # Visualise a few digits
    fig, axes = plt.subplots(1, 8, figsize=(12, 2))
    for i, ax in enumerate(axes):
        ax.imshow(X_train[i], cmap='gray')
        ax.set_title(f'Label: {y_train[i]}')
        ax.axis('off')
    plt.suptitle('Sample MNIST digits', fontweight='bold')
    plt.tight_layout()
    plt.savefig('mnist_samples.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("""
    Code to run in Colab:
    
    (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
    X_train = X_train.reshape(-1, 784) / 255.0
    X_test  = X_test.reshape(-1, 784) / 255.0
    """)

In [ ]:
if TF_AVAILABLE:
    # Build the model
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(784,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')   # 10 classes
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    model.summary()
    
    # Count parameters
    total_params = model.count_params()
    print(f'\nTotal trainable parameters: {total_params:,}')
    print(f'Layer 1: 784 inputs x 128 neurons + 128 biases = {784*128 + 128:,}')
    print(f'Layer 2: 128 inputs x 64 neurons + 64 biases  = {128*64 + 64:,}')
    print(f'Layer 3: 64 inputs x 10 neurons + 10 biases   = {64*10 + 10:,}')
else:
    print("""
    Model architecture:
    
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(784,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    """)

In [ ]:
if TF_AVAILABLE:
    # Train
    history = model.fit(
        X_train_flat, y_train,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )
    
    # Evaluate
    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    print(f'\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
    print(f'Test loss:     {test_loss:.4f}')
    
    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history.history['accuracy'], label='Train', color='#0066CC', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Validation', color='#FF6600', linewidth=2)
    ax1.set_title('Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(history.history['loss'], label='Train', color='#0066CC', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Validation', color='#FF6600', linewidth=2)
    ax2.set_title('Loss', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('mnist_training_curves.png', dpi=100, bbox_inches='tight')
    plt.show()

## Part 8: Real World Problem

### The Problem Engineers Actually Face: Vanishing Gradients in Production

A team at a startup builds a 10-layer neural network to classify customer support tickets into 50 categories. They use sigmoid activations throughout. After 100 epochs, training accuracy is stuck at 42%.

The first 7 layers are barely learning. Their gradients are ~0.0001 by the time backpropagation reaches them.

**Why:** Sigmoid's max derivative is 0.25. Chain rule multiplies gradients layer by layer. After 7 layers: `0.25^7 = 0.00006` — effectively zero. Weights in early layers stop updating.

**The fix:** Switch hidden layer activations from sigmoid to ReLU. ReLU's derivative is exactly 1 for positive inputs. No gradient shrinkage through layers.

Run the cell below to see the gradient magnitude difference.

In [ ]:
# Demonstrate vanishing gradients
n_layers = 10
x_input = np.array([0.5])  # typical post-activation value

# Sigmoid gradient after N layers
sigmoid_grad = 1.0
relu_grad = 1.0

sigmoid_grads = []
relu_grads = []

for layer in range(1, n_layers + 1):
    sigmoid_grad *= 0.25   # max sigmoid derivative
    relu_grad    *= 1.0    # ReLU derivative (for positive inputs)
    sigmoid_grads.append(sigmoid_grad)
    relu_grads.append(relu_grad)

print('=== GRADIENT MAGNITUDE AFTER EACH LAYER ===')
print(f'{"Layer":<8} {"Sigmoid":<20} {"ReLU"}')
print('-' * 40)
for i, (s, r) in enumerate(zip(sigmoid_grads, relu_grads)):
    print(f'{i+1:<8} {s:<20.8f} {r:.1f}')

print()
print(f'After 10 layers:')
print(f'  Sigmoid gradient: {sigmoid_grads[-1]:.10f} (essentially zero)')
print(f'  ReLU gradient:    {relu_grads[-1]:.1f} (unchanged)')
print()
print('Solution: Use ReLU in hidden layers. Only use sigmoid at the output for binary classification.')

## Part 9: Practice Exercise

**Task:** Modify the XOR network from Part 5 to solve a different problem: the AND gate.

| Input 1 | Input 2 | AND output |
|---------|---------|------------|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Questions to answer:
1. Can a single-layer network (no hidden layer) solve AND? Why/why not?
2. How few neurons do you need in the hidden layer to solve AND?
3. What happens to your training loss if you increase the learning rate to 5.0?

In [ ]:
# YOUR CODE HERE
# Define AND gate data
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([[0], [0], [0], [1]])   # AND output

# Hint: AND IS linearly separable (unlike XOR), so a single layer works
# Try both: 1 layer (direct sigmoid) and 2 layers
# Compare final loss and number of epochs needed

# ====== SOLUTION (remove this comment and try it yourself first) ======
# AND IS linearly separable, so you can solve it without a hidden layer
# But our 2-layer net will still solve it, just less efficiently

np.random.seed(0)
W1_and = np.random.randn(2, 4) * 0.1
b1_and = np.zeros((1, 4))
W2_and = np.random.randn(4, 1) * 0.1
b2_and = np.zeros((1, 1))

lr = 0.5
losses_and = []

for epoch in range(5000):
    Z1 = X_and @ W1_and + b1_and
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2_and + b2_and
    A2 = sigmoid(Z2)
    
    loss = binary_crossentropy(y_and, A2)
    losses_and.append(loss)
    
    dA2 = -(y_and / (A2 + 1e-8) - (1 - y_and) / (1 - A2 + 1e-8))
    dZ2 = dA2 * sigmoid_deriv(Z2)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ W2_and.T
    dZ1 = dA1 * sigmoid_deriv(Z1)
    dW1 = X_and.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    W2_and -= lr * dW2
    b2_and -= lr * db2
    W1_and -= lr * dW1
    b1_and -= lr * db1

print('AND gate predictions after training:')
for xi, yi, pred in zip(X_and, y_and.flatten(), A2.flatten()):
    status = 'CORRECT' if round(pred) == yi else 'WRONG'
    print(f'  {xi} -> Expected {yi} | Got {pred:.4f} | {status}')
print(f'Final loss: {losses_and[-1]:.6f}')

## Summary

| Concept | What You Learned |
|---------|------------------|
| Network structure | Layers, weights, biases, activations |
| Forward pass | Z = X @ W + b, A = activation(Z) |
| Activation functions | Sigmoid, ReLU, Tanh and when to use each |
| Loss | Binary cross-entropy penalises wrong confident predictions |
| Backpropagation | Chain rule computes gradients layer by layer |
| Vanishing gradients | Sigmoid kills gradients in deep nets; ReLU fixes it |
| Keras | Same operations, abstracted for production use |

**Tomorrow — Day 24: CNNs**  
How convolutional layers detect edges, shapes, and objects in images. Why a regular dense network fails on image data at scale.

---

**GitHub repo:** [42-days-aiml-challenge](https://github.com/VaishnaviJagtap18/42-days-aiml-challenge)  
**Connect on LinkedIn:** [Vaishnavi Jagtap](https://www.linkedin.com/in/vaishnavi-jagtap18)

#42DaysOfML #MachineLearning #NeuralNetworks #DeepLearning #Python